# MovieLens-100K Core Models (Single Notebook)

This notebook runs core recommender models for MovieLens-100K in one place: data preparation, model training, evaluation, and artifact export.

It includes local in-notebook implementations of:
- Session-based baselines (Popularity, SKNN variants)
- Neural sequence models (GRU4Rec, BERT4Rec, SASRec)
- Embedding-based models (LLMSeqSim, LLM2GRU4Rec, LLM2BERT4Rec, LLM2SASRec)

Note: LLM-style models require item embeddings. In this notebook, embeddings are derived from MovieLens genre indicators so everything can run self-contained.

In [ ]:
from __future__ import annotations

import json
import os
import pickle
import subprocess
import sys
import tarfile
import time
import urllib.request
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

REPO_URL = "https://github.com/faroukq1/LLM-Sequential-Recommendation.git"
KAGGLE_REPO_DIR = Path("/kaggle/working/LLM-Sequential-Recommendation")


def is_project_root(path: Path) -> bool:
    return (path / "main").is_dir() and (path / "kaggle").is_dir()


def find_project_root(start: Path) -> Path:
    env_root = os.environ.get("PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if is_project_root(candidate):
            return candidate

    for candidate in [start, *start.parents]:
        if is_project_root(candidate):
            return candidate

    common_candidates = [
        KAGGLE_REPO_DIR,
        Path("/home/farouk/files/LLM-Sequential-Recommendation"),
    ]
    for candidate in common_candidates:
        if candidate.exists() and is_project_root(candidate):
            return candidate

    if Path("/kaggle/working").exists():
        if not KAGGLE_REPO_DIR.exists():
            print("Project root not found. Cloning repository into /kaggle/working ...")
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(KAGGLE_REPO_DIR)],
                check=True,
            )
        if is_project_root(KAGGLE_REPO_DIR):
            return KAGGLE_REPO_DIR

    raise FileNotFoundError(
        "Could not locate project root containing 'main' and 'kaggle'. "
        "Set PROJECT_ROOT to your repository path.",
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
from main.data.session_dataset import SessionDataset
from main.data.temporal_split import TemporalSplit
from main.eval.evaluation import Evaluation
from main.eval.metrics.catalog_coverage import CatalogCoverage
from main.eval.metrics.hitrate import HitRate
from main.eval.metrics.metric import MetricDependency
from main.eval.metrics.mrr import MeanReciprocalRank
from main.eval.metrics.ndcg import NormalizedDiscountedCumulativeGain

import tensorflow as tf
from tensorflow import keras

DATASET_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"

WORK_DIR = PROJECT_ROOT / "kaggle" / "artifacts" / "ml100k_core_models_single_notebook"
DATA_DIR = WORK_DIR / "data"
RESULTS_DIR = WORK_DIR / "results"
RECS_DIR = WORK_DIR / "recommendations"
for directory in [DATA_DIR, RESULTS_DIR, RECS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

INCLUDE_MODELS = [
    "Popular",
    "SKNN",
    "S-SKNN",
    "SF-SKNN",
    "V-SKNN",
    "GRU4Rec",
    "BERT4Rec",
    "SASRec",
    "LLMSeqSim",
    "LLM2GRU4Rec",
    "LLM2BERT4Rec",
    "LLM2SASRec",
]

TOP_KS = [10, 20]

SESSION_GAP_MINUTES = 30
MIN_SESSION_LEN = 2
TEST_FRAC = 0.2
NUM_FOLDS = 0
FILTER_NON_TRAINED_TEST_ITEMS = True

CORES = max(1, min(4, os.cpu_count() or 1))
EVAL_CORES = 1
VERBOSE = True

SKNN_K = 200
SKNN_SAMPLE_SIZE = 1000

MAX_SEQ_LEN = 20
BASE_NEURAL_EMB_DIM = 64
EMBED_NEURAL_EMB_DIM = 20
HIDDEN_DIM = 128
DROP_RATE = 0.2
NUM_EPOCHS = 3
FIT_BATCH_SIZE = 256
PRED_BATCH_SIZE = 1024
TRAIN_VAL_FRACTION = 0.1
EARLY_STOPPING_PATIENCE = 2
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
MASK_PROB = 0.4

print("Work dir:", WORK_DIR)
print("Models:", INCLUDE_MODELS)

In [ ]:
from collections import defaultdict


def _coerce_embedding(value: Any) -> np.ndarray:
    if isinstance(value, np.ndarray):
        return value.astype(np.float32)
    if isinstance(value, list):
        return np.asarray(value, dtype=np.float32)
    if isinstance(value, str):
        return np.asarray(json.loads(value), dtype=np.float32)
    raise TypeError(f"Unsupported embedding type: {type(value)}")


def _pad_left(values: list[int], length: int, pad_value: int = 0) -> np.ndarray:
    if len(values) >= length:
        return np.asarray(values[-length:], dtype=np.int32)
    pad_size = length - len(values)
    return np.asarray(([pad_value] * pad_size) + values, dtype=np.int32)


def _top_k_indices(scores: np.ndarray, top_k: int) -> np.ndarray:
    if top_k <= 0:
        return np.asarray([], dtype=np.int64)
    top_k = min(top_k, len(scores))
    idx = np.argpartition(scores, -top_k)[-top_k:]
    return idx[np.argsort(scores[idx])[::-1]]


class BaseModel:
    def __init__(self, is_verbose: bool = False, cores: int = 1) -> None:
        self.is_verbose = is_verbose
        self.cores = cores
        self.is_trained = False

    def train(self, train_data: pd.DataFrame, item_data: pd.DataFrame | None = None) -> None:
        raise NotImplementedError

    def predict(
        self,
        predict_data: dict[int, np.ndarray],
        top_k: int = 10,
        return_scores: bool = False,
    ):
        raise NotImplementedError

    def name(self) -> str:
        raise NotImplementedError


class SessionBasedPopular(BaseModel):
    def __init__(self, is_verbose: bool = False, cores: int = 1, filter_prompt_items: bool = True) -> None:
        super().__init__(is_verbose=is_verbose, cores=cores)
        self.filter_prompt_items = filter_prompt_items
        self.popular_items: list[int] = []

    def train(self, train_data: pd.DataFrame, item_data: pd.DataFrame | None = None) -> None:
        self.popular_items = train_data["ItemId"].value_counts().index.astype(int).tolist()
        self.is_trained = True

    def predict(
        self,
        predict_data: dict[int, np.ndarray],
        top_k: int = 10,
        return_scores: bool = False,
    ):
        predictions: dict[int, np.ndarray] = {}
        score_map: dict[int, np.ndarray] = {}
        for case_id, prompt_items in predict_data.items():
            seen = set(int(x) for x in prompt_items)
            recs: list[int] = []
            rec_scores: list[float] = []
            for rank, item in enumerate(self.popular_items):
                if self.filter_prompt_items and item in seen:
                    continue
                recs.append(item)
                rec_scores.append(float(len(self.popular_items) - rank))
                if len(recs) == top_k:
                    break
            predictions[int(case_id)] = np.asarray(recs, dtype=int)
            score_map[int(case_id)] = np.asarray(rec_scores, dtype=np.float32)
        if return_scores:
            return predictions, score_map
        return predictions

    def name(self) -> str:
        return "Popularity"


class SessionBasedCF(BaseModel):
    def __init__(
        self,
        is_verbose: bool = False,
        cores: int = 1,
        k: int = 100,
        sample_size: int = 500,
        sampling: str = "recent",
        use_item_embeddings: bool = False,
        prompt_session_emb_comb_strategy: str = "last",
        training_session_emb_comb_strategy: str = "last",
        dim_reduction_config: dict | None = None,
        sequential_weighting: bool = False,
        sequential_filter: bool = False,
        decay: str | None = None,
        training_session_decay: str | None = None,
        similarity_measure: str = "cosine",
        filter_prompt_items: bool = True,
        last_n_items: int | None = None,
        idf_weighting: bool = False,
        **_: dict,
    ) -> None:
        super().__init__(is_verbose=is_verbose, cores=cores)
        self.k = int(k)
        self.sample_size = int(sample_size)
        self.sampling = sampling
        self.use_item_embeddings = use_item_embeddings
        self.prompt_session_emb_comb_strategy = prompt_session_emb_comb_strategy
        self.training_session_emb_comb_strategy = training_session_emb_comb_strategy
        self.dim_reduction_config = dim_reduction_config
        self.sequential_weighting = sequential_weighting
        self.sequential_filter = sequential_filter
        self.decay = decay
        self.training_session_decay = training_session_decay
        self.similarity_measure = similarity_measure
        self.filter_prompt_items = filter_prompt_items
        self.last_n_items = last_n_items
        self.idf_weighting = idf_weighting

        self.session_items: dict[int, np.ndarray] = {}
        self.item_to_sessions: dict[int, list[int]] = defaultdict(list)
        self.popular_items: list[int] = []
        self.idf: dict[int, float] = {}
        self.item_embedding_lookup: dict[int, np.ndarray] | None = None

    def _session_embedding(self, items: np.ndarray) -> np.ndarray | None:
        if self.item_embedding_lookup is None:
            return None
        vectors = [self.item_embedding_lookup[int(i)] for i in items if int(i) in self.item_embedding_lookup]
        if not vectors:
            return None
        mat = np.vstack(vectors).astype(np.float32)
        if self.training_session_emb_comb_strategy == "mean":
            return mat.mean(axis=0)
        return mat[-1]

    def _cosine(self, a: np.ndarray, b: np.ndarray) -> float:
        denom = (np.linalg.norm(a) * np.linalg.norm(b))
        if denom <= 0:
            return 0.0
        return float(np.dot(a, b) / denom)

    def _id_based_similarity(self, prompt: np.ndarray, candidate: np.ndarray) -> float:
        prompt_set = set(int(i) for i in prompt)
        candidate_set = set(int(i) for i in candidate)
        common = prompt_set.intersection(candidate_set)
        if not common:
            return 0.0

        weight_lookup: dict[int, float] = {}
        if self.sequential_weighting or self.decay is not None:
            n = len(prompt)
            for pos, item in enumerate(prompt):
                item = int(item)
                recency = (pos + 1) / max(n, 1)
                if self.decay == "harmonic":
                    weight = 1.0 / max(1, (n - pos))
                elif self.decay == "linear":
                    weight = recency
                elif self.decay == "quadratic":
                    weight = recency * recency
                elif self.decay == "log":
                    weight = float(np.log1p(pos + 1))
                else:
                    weight = recency
                weight_lookup[item] = max(weight_lookup.get(item, 0.0), weight)
            num = sum(weight_lookup.get(item, 1.0) for item in common)
        else:
            num = float(len(common))

        den = np.sqrt(max(len(prompt_set), 1) * max(len(candidate_set), 1))
        return float(num / den)

    def _session_similarity(self, prompt: np.ndarray, candidate_sid: int) -> float:
        candidate_items = self.session_items[candidate_sid]
        if self.use_item_embeddings and self.item_embedding_lookup is not None:
            prompt_vec = self._session_embedding(prompt)
            candidate_vec = self._session_embedding(candidate_items)
            if prompt_vec is None or candidate_vec is None:
                return 0.0
            if self.similarity_measure == "dot":
                return float(np.dot(prompt_vec, candidate_vec))
            return self._cosine(prompt_vec, candidate_vec)
        return self._id_based_similarity(prompt, candidate_items)

    def train(self, train_data: pd.DataFrame, item_data: pd.DataFrame | None = None) -> None:
        train_df = train_data.copy()
        sort_cols = ["SessionId", "ItemId"]
        if "Time" in train_df.columns:
            sort_cols = ["SessionId", "Time", "ItemId"]
        train_df = train_df.sort_values(sort_cols).reset_index(drop=True)

        self.session_items = {
            int(sid): grp["ItemId"].astype(int).to_numpy()
            for sid, grp in train_df.groupby("SessionId", sort=False)
        }

        self.item_to_sessions = defaultdict(list)
        for sid, items in self.session_items.items():
            for item in np.unique(items):
                self.item_to_sessions[int(item)].append(int(sid))

        self.popular_items = train_df["ItemId"].value_counts().index.astype(int).tolist()

        if self.idf_weighting:
            num_sessions = max(len(self.session_items), 1)
            self.idf = {
                int(item): float(np.log((1 + num_sessions) / (1 + len(set(sids)))) + 1.0)
                for item, sids in self.item_to_sessions.items()
            }
        else:
            self.idf = {}

        if self.use_item_embeddings and item_data is not None:
            tmp = item_data.copy()
            tmp["embedding"] = tmp["embedding"].apply(_coerce_embedding)
            self.item_embedding_lookup = {
                int(row["ItemId"]): row["embedding"] for _, row in tmp.iterrows()
            }
        else:
            self.item_embedding_lookup = None

        self.is_trained = True

    def predict(
        self,
        predict_data: dict[int, np.ndarray],
        top_k: int = 10,
        return_scores: bool = False,
    ):
        if not self.is_trained:
            raise RuntimeError("SessionBasedCF is not trained.")

        predictions: dict[int, np.ndarray] = {}
        score_map: dict[int, np.ndarray] = {}

        for case_id, prompt_arr in predict_data.items():
            prompt_items = np.asarray(prompt_arr, dtype=int)
            if self.last_n_items is not None and self.last_n_items > 0:
                prompt_items = prompt_items[-self.last_n_items:]

            candidate_sids: set[int] = set()
            for item in np.unique(prompt_items):
                candidate_sids.update(self.item_to_sessions.get(int(item), []))

            if not candidate_sids:
                candidate_sids = set(self.session_items.keys())

            if len(candidate_sids) > self.sample_size:
                ordered = sorted(candidate_sids)
                if self.sampling == "recent":
                    candidate_sids = set(ordered[-self.sample_size:])
                else:
                    candidate_sids = set(ordered[: self.sample_size])

            similarities: list[tuple[int, float]] = []
            for sid in candidate_sids:
                sim = self._session_similarity(prompt_items, sid)
                if sim > 0:
                    similarities.append((sid, sim))

            similarities.sort(key=lambda x: x[1], reverse=True)
            neighbors = similarities[: self.k]

            seen = set(int(x) for x in prompt_items)
            scores = defaultdict(float)

            for sid, sim in neighbors:
                candidate_items = self.session_items[sid]
                if self.sequential_filter and len(prompt_items) > 0:
                    last_prompt_item = int(prompt_items[-1])
                    positions = np.where(candidate_items == last_prompt_item)[0]
                    if len(positions) > 0:
                        candidate_items = candidate_items[positions.max() + 1 :]

                for item in candidate_items:
                    item = int(item)
                    if self.filter_prompt_items and item in seen:
                        continue
                    score = float(sim)
                    if self.idf_weighting:
                        score *= self.idf.get(item, 1.0)
                    scores[item] += score

            ranked_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)
            recs = [item for item, _ in ranked_items[:top_k]]
            rec_scores = [float(score) for _, score in ranked_items[:top_k]]

            if len(recs) < top_k:
                for item in self.popular_items:
                    if self.filter_prompt_items and item in seen:
                        continue
                    if item in recs:
                        continue
                    recs.append(int(item))
                    rec_scores.append(0.0)
                    if len(recs) == top_k:
                        break

            predictions[int(case_id)] = np.asarray(recs[:top_k], dtype=int)
            score_map[int(case_id)] = np.asarray(rec_scores[:top_k], dtype=np.float32)

        if return_scores:
            return predictions, score_map
        return predictions

    def name(self) -> str:
        if self.use_item_embeddings:
            return "SKNN_EMB"
        if self.decay is not None:
            return "V-SKNN"
        if self.sequential_filter:
            return "SF-SKNN"
        if self.sequential_weighting:
            return "S-SKNN"
        return "SKNN"


class NeuralSequenceRecommender(BaseModel):
    def __init__(
        self,
        model_name: str,
        architecture: str,
        N: int = 20,
        L: int = 1,
        h: int = 2,
        emb_dim: int = 64,
        hidden_dim: int = 128,
        drop_rate: float = 0.2,
        mask_prob: float = 0.4,
        num_epochs: int = 3,
        fit_batch_size: int = 256,
        pred_batch_size: int = 1024,
        train_val_fraction: float = 0.1,
        early_stopping_patience: int = 2,
        learning_rate: float = 1e-3,
        weight_decay: float = 0.0,
        optimizer_kwargs: dict | None = None,
        activation: str = "relu",
        pred_seen: bool = False,
        is_verbose: bool = False,
        cores: int = 1,
        transformer_layer_kwargs: dict | None = None,
        **_: dict,
    ) -> None:
        super().__init__(is_verbose=is_verbose, cores=cores)
        self.model_name = model_name
        self.architecture = architecture
        self.N = int(N)
        self.L = int(L)
        self.h = int(h)
        self.emb_dim = int(emb_dim)
        self.hidden_dim = int(hidden_dim)
        self.drop_rate = float(drop_rate)
        self.mask_prob = float(mask_prob)
        self.num_epochs = int(num_epochs)
        self.fit_batch_size = int(fit_batch_size)
        self.pred_batch_size = int(pred_batch_size)
        self.train_val_fraction = float(train_val_fraction)
        self.early_stopping_patience = int(early_stopping_patience)
        self.learning_rate = float(learning_rate)
        self.weight_decay = float(weight_decay)
        self.optimizer_kwargs = optimizer_kwargs or {}
        self.activation = activation
        self.pred_seen = pred_seen
        self.transformer_layer_kwargs = transformer_layer_kwargs or {}

        self.model: keras.Model | None = None
        self.item_to_idx: dict[int, int] = {}
        self.idx_to_item: dict[int, int] = {}
        self.popular_items: list[int] = []

    def _build_initial_embedding_matrix(self) -> np.ndarray | None:
        return None

    def _build_training_examples(self, train_df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
        x_rows: list[np.ndarray] = []
        y_rows: list[int] = []
        sort_cols = ["SessionId", "ItemId"]
        if "Time" in train_df.columns:
            sort_cols = ["SessionId", "Time", "ItemId"]
        ordered = train_df.sort_values(sort_cols).reset_index(drop=True)

        for _, session_df in ordered.groupby("SessionId", sort=False):
            idx_items = [self.item_to_idx[int(i)] for i in session_df["ItemId"].tolist() if int(i) in self.item_to_idx]
            if len(idx_items) < 2:
                continue
            for t in range(1, len(idx_items)):
                seq = idx_items[max(0, t - self.N) : t]
                x_rows.append(_pad_left(seq, self.N, pad_value=0))
                y_rows.append(idx_items[t])

        if not x_rows:
            raise ValueError("Not enough sequence data to create training samples.")

        return np.vstack(x_rows).astype(np.int32), np.asarray(y_rows, dtype=np.int32)

    def _build_model(self, num_items_plus_pad: int, initial_embeddings: np.ndarray | None = None) -> keras.Model:
        inputs = keras.Input(shape=(self.N,), dtype="int32")
        emb_layer = keras.layers.Embedding(
            input_dim=num_items_plus_pad,
            output_dim=self.emb_dim,
            mask_zero=True,
            name="item_emb",
        )
        x = emb_layer(inputs)

        if self.architecture == "gru":
            x = keras.layers.Dropout(self.drop_rate)(x)
            x = keras.layers.GRU(self.hidden_dim)(x)
            x = keras.layers.Dropout(self.drop_rate)(x)
        else:
            pos_indices = tf.range(start=0, limit=self.N, delta=1)
            pos_emb = keras.layers.Embedding(self.N, self.emb_dim, name="pos_emb")(pos_indices)
            x = x + pos_emb
            for _ in range(max(1, self.L)):
                attn = keras.layers.MultiHeadAttention(
                    num_heads=max(1, self.h),
                    key_dim=max(4, self.emb_dim // max(1, self.h)),
                    dropout=self.drop_rate,
                )(x, x)
                x = keras.layers.LayerNormalization(epsilon=1e-6)(x + attn)
                ff = keras.layers.Dense(self.hidden_dim, activation=self.activation)(x)
                ff = keras.layers.Dropout(self.drop_rate)(ff)
                ff = keras.layers.Dense(self.emb_dim)(ff)
                x = keras.layers.LayerNormalization(epsilon=1e-6)(x + ff)
            if self.architecture == "bert":
                x = keras.layers.GlobalAveragePooling1D()(x)
            else:
                x = keras.layers.Lambda(lambda t: t[:, -1, :])(x)

        outputs = keras.layers.Dense(num_items_plus_pad, activation="softmax")(x)
        model = keras.Model(inputs=inputs, outputs=outputs)

        if initial_embeddings is not None:
            emb_layer.build((None, self.N))
            emb_layer.set_weights([initial_embeddings.astype(np.float32)])

        lr = float(self.optimizer_kwargs.get("learning_rate", self.learning_rate))
        wd = float(self.optimizer_kwargs.get("weight_decay", self.weight_decay))
        if wd > 0 and hasattr(keras.optimizers, "AdamW"):
            optimizer = keras.optimizers.AdamW(learning_rate=lr, weight_decay=wd)
        else:
            optimizer = keras.optimizers.Adam(learning_rate=lr)

        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
        )
        return model

    def train(self, train_data: pd.DataFrame, item_data: pd.DataFrame | None = None) -> None:
        tf.keras.utils.set_random_seed(42)

        unique_items = sorted(train_data["ItemId"].astype(int).unique().tolist())
        self.item_to_idx = {item: idx + 1 for idx, item in enumerate(unique_items)}
        self.idx_to_item = {idx: item for item, idx in self.item_to_idx.items()}
        self.popular_items = train_data["ItemId"].value_counts().index.astype(int).tolist()

        x_train, y_train = self._build_training_examples(train_data)

        initial_embeddings = self._build_initial_embedding_matrix()
        self.model = self._build_model(
            num_items_plus_pad=len(self.item_to_idx) + 1,
            initial_embeddings=initial_embeddings,
        )

        callbacks: list[keras.callbacks.Callback] = []
        if self.early_stopping_patience > 0:
            callbacks.append(
                keras.callbacks.EarlyStopping(
                    monitor="loss",
                    patience=self.early_stopping_patience,
                    restore_best_weights=True,
                )
            )

        self.model.fit(
            x_train,
            y_train,
            epochs=self.num_epochs,
            batch_size=min(self.fit_batch_size, len(x_train)),
            verbose=1 if self.is_verbose else 0,
            callbacks=callbacks,
        )
        self.is_trained = True

    def predict(
        self,
        predict_data: dict[int, np.ndarray],
        top_k: int = 10,
        return_scores: bool = False,
    ):
        if not self.is_trained or self.model is None:
            raise RuntimeError(f"{self.model_name} is not trained.")

        case_ids = list(predict_data.keys())
        x_input = np.vstack(
            [
                _pad_left(
                    [self.item_to_idx[int(i)] for i in predict_data[cid] if int(i) in self.item_to_idx],
                    self.N,
                    pad_value=0,
                )
                for cid in case_ids
            ]
        ).astype(np.int32)

        probs = self.model.predict(
            x_input,
            batch_size=min(self.pred_batch_size, len(case_ids)),
            verbose=0,
        )

        predictions: dict[int, np.ndarray] = {}
        score_map: dict[int, np.ndarray] = {}

        for row_idx, case_id in enumerate(case_ids):
            prompt_items = [int(i) for i in predict_data[case_id]]
            seen = set(prompt_items)

            row_scores = probs[row_idx].astype(np.float64)
            row_scores[0] = -np.inf
            if not self.pred_seen:
                for item in seen:
                    item_idx = self.item_to_idx.get(item)
                    if item_idx is not None:
                        row_scores[item_idx] = -np.inf

            top_idx = _top_k_indices(row_scores, top_k)
            recs = [self.idx_to_item[int(i)] for i in top_idx if int(i) in self.idx_to_item]
            rec_scores = [float(row_scores[int(i)]) for i in top_idx if int(i) in self.idx_to_item]

            if len(recs) < top_k:
                for item in self.popular_items:
                    if (not self.pred_seen) and item in seen:
                        continue
                    if item in recs:
                        continue
                    recs.append(int(item))
                    rec_scores.append(0.0)
                    if len(recs) == top_k:
                        break

            predictions[int(case_id)] = np.asarray(recs[:top_k], dtype=int)
            score_map[int(case_id)] = np.asarray(rec_scores[:top_k], dtype=np.float32)

        if return_scores:
            return predictions, score_map
        return predictions

    def name(self) -> str:
        return self.model_name


class GRURec(NeuralSequenceRecommender):
    def __init__(self, **kwargs: dict) -> None:
        super().__init__(model_name="GRU4Rec", architecture="gru", **kwargs)


class BERT(NeuralSequenceRecommender):
    def __init__(self, **kwargs: dict) -> None:
        super().__init__(model_name="BERT4Rec", architecture="bert", **kwargs)


class SASRec(NeuralSequenceRecommender):
    def __init__(self, **kwargs: dict) -> None:
        super().__init__(model_name="SASRec", architecture="sas", **kwargs)


class LLMSeqSim(BaseModel):
    def __init__(
        self,
        is_verbose: bool = False,
        cores: int = 1,
        similarity_measure: str = "cosine",
        embedding_combination_strategy: str = "last",
        combination_decay: str | None = None,
        filter_prompt_items: bool = True,
        batch_size: int = 1024,
        dim_reduction_config: dict | None = None,
        **_: dict,
    ) -> None:
        super().__init__(is_verbose=is_verbose, cores=cores)
        self.similarity_measure = similarity_measure
        self.embedding_combination_strategy = embedding_combination_strategy
        self.combination_decay = combination_decay
        self.filter_prompt_items = filter_prompt_items
        self.batch_size = batch_size
        self.dim_reduction_config = dim_reduction_config

        self.item_ids: np.ndarray | None = None
        self.item_embeddings: np.ndarray | None = None
        self.item_embeddings_norm: np.ndarray | None = None
        self.item_to_index: dict[int, int] = {}
        self.popular_items: list[int] = []

    def train(self, train_data: pd.DataFrame, item_data: pd.DataFrame | None = None) -> None:
        if item_data is None:
            raise ValueError("LLMSeqSim requires item_data with an embedding column.")

        item_df = item_data.copy()
        item_df["embedding"] = item_df["embedding"].apply(_coerce_embedding)

        self.item_ids = item_df["ItemId"].astype(int).to_numpy()
        self.item_embeddings = np.vstack(item_df["embedding"].values).astype(np.float32)
        norms = np.linalg.norm(self.item_embeddings, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        self.item_embeddings_norm = self.item_embeddings / norms
        self.item_to_index = {int(item): idx for idx, item in enumerate(self.item_ids)}
        self.popular_items = train_data["ItemId"].value_counts().index.astype(int).tolist()
        self.is_trained = True

    def _combine_session(self, prompt_items: np.ndarray) -> np.ndarray | None:
        vectors = [
            self.item_embeddings[self.item_to_index[int(i)]]
            for i in prompt_items
            if int(i) in self.item_to_index
        ]
        if not vectors:
            return None
        mat = np.vstack(vectors).astype(np.float32)
        if self.embedding_combination_strategy == "mean":
            return mat.mean(axis=0)
        return mat[-1]

    def _score(self, query: np.ndarray) -> np.ndarray:
        if self.similarity_measure == "dot":
            return self.item_embeddings @ query
        if self.similarity_measure == "euclidean":
            return -np.linalg.norm(self.item_embeddings - query.reshape(1, -1), axis=1)
        q_norm = query / max(float(np.linalg.norm(query)), 1e-12)
        return self.item_embeddings_norm @ q_norm

    def predict(
        self,
        predict_data: dict[int, np.ndarray],
        top_k: int = 10,
        return_scores: bool = False,
    ):
        if not self.is_trained or self.item_embeddings is None or self.item_ids is None:
            raise RuntimeError("LLMSeqSim is not trained.")

        predictions: dict[int, np.ndarray] = {}
        score_map: dict[int, np.ndarray] = {}

        for case_id, prompt_items in predict_data.items():
            prompt_items = np.asarray(prompt_items, dtype=int)
            seen = set(int(i) for i in prompt_items)
            query = self._combine_session(prompt_items)

            if query is None:
                recs = []
                rec_scores = []
            else:
                scores = self._score(query).astype(np.float64)
                if self.filter_prompt_items:
                    for item in seen:
                        idx = self.item_to_index.get(item)
                        if idx is not None:
                            scores[idx] = -np.inf

                top_idx = _top_k_indices(scores, top_k)
                recs = [int(self.item_ids[idx]) for idx in top_idx]
                rec_scores = [float(scores[idx]) for idx in top_idx]

            if len(recs) < top_k:
                for item in self.popular_items:
                    if self.filter_prompt_items and item in seen:
                        continue
                    if item in recs:
                        continue
                    recs.append(int(item))
                    rec_scores.append(0.0)
                    if len(recs) == top_k:
                        break

            predictions[int(case_id)] = np.asarray(recs[:top_k], dtype=int)
            score_map[int(case_id)] = np.asarray(rec_scores[:top_k], dtype=np.float32)

        if return_scores:
            return predictions, score_map
        return predictions

    def name(self) -> str:
        return "LLMSeqSim"


class _EmbeddingBootstrappedNeural(NeuralSequenceRecommender):
    def __init__(
        self,
        product_embeddings_location: str,
        red_method: str = "PCA",
        red_params: dict | None = None,
        **kwargs: dict,
    ) -> None:
        self.product_embeddings_location = product_embeddings_location
        self.red_method = red_method
        self.red_params = red_params or {}
        self.external_embeddings = self._load_external_embeddings(product_embeddings_location)
        super().__init__(**kwargs)

    @staticmethod
    def _load_external_embeddings(path: str) -> dict[int, np.ndarray]:
        df = pd.read_csv(path)
        lookup: dict[int, np.ndarray] = {}
        for _, row in df.iterrows():
            lookup[int(row["ItemId"])] = _coerce_embedding(row["embedding"])
        return lookup

    def _build_initial_embedding_matrix(self) -> np.ndarray | None:
        matrix = np.random.normal(
            loc=0.0,
            scale=0.01,
            size=(len(self.item_to_idx) + 1, self.emb_dim),
        ).astype(np.float32)
        matrix[0] = 0.0
        for item, idx in self.item_to_idx.items():
            vec = self.external_embeddings.get(int(item))
            if vec is None:
                continue
            if vec.shape[0] < self.emb_dim:
                vec = np.pad(vec, (0, self.emb_dim - vec.shape[0]), mode="constant")
            elif vec.shape[0] > self.emb_dim:
                vec = vec[: self.emb_dim]
            matrix[idx] = vec.astype(np.float32)
        return matrix


class GRURecWithEmbeddings(_EmbeddingBootstrappedNeural):
    def __init__(self, **kwargs: dict) -> None:
        super().__init__(model_name="LLM2GRU4Rec", architecture="gru", **kwargs)


class BERTWithEmbeddings(_EmbeddingBootstrappedNeural):
    def __init__(self, **kwargs: dict) -> None:
        super().__init__(model_name="LLM2BERT4Rec", architecture="bert", **kwargs)


class SASRecWithEmbeddings(_EmbeddingBootstrappedNeural):
    def __init__(self, **kwargs: dict) -> None:
        super().__init__(model_name="LLM2SASRec", architecture="sas", **kwargs)


print("Loaded local notebook model implementations.")

In [ ]:
# Fix GRU mask + cuDNN incompatibility on GPU by disabling cuDNN for GRU models.
# This patch is idempotent and safe to re-run.

if not hasattr(NeuralSequenceRecommender, "_original_build_model"):
    NeuralSequenceRecommender._original_build_model = NeuralSequenceRecommender._build_model


def _patched_build_model(self, num_items_plus_pad: int, initial_embeddings: np.ndarray | None = None) -> keras.Model:
    if self.architecture != "gru":
        return NeuralSequenceRecommender._original_build_model(
            self, num_items_plus_pad, initial_embeddings
        )

    inputs = keras.Input(shape=(self.N,), dtype="int32")
    emb_layer = keras.layers.Embedding(
        input_dim=num_items_plus_pad,
        output_dim=self.emb_dim,
        mask_zero=True,
        name="item_emb",
    )
    x = emb_layer(inputs)
    x = keras.layers.Dropout(self.drop_rate)(x)

    try:
        gru_layer = keras.layers.GRU(self.hidden_dim, use_cudnn=False)
    except TypeError:
        gru_layer = keras.layers.GRU(self.hidden_dim)

    x = gru_layer(x)
    x = keras.layers.Dropout(self.drop_rate)(x)
    outputs = keras.layers.Dense(num_items_plus_pad, activation="softmax")(x)
    model = keras.Model(inputs=inputs, outputs=outputs)

    if initial_embeddings is not None:
        emb_layer.build((None, self.N))
        emb_layer.set_weights([initial_embeddings.astype(np.float32)])

    lr = float(self.optimizer_kwargs.get("learning_rate", self.learning_rate))
    wd = float(self.optimizer_kwargs.get("weight_decay", self.weight_decay))
    if wd > 0 and hasattr(keras.optimizers, "AdamW"):
        optimizer = keras.optimizers.AdamW(learning_rate=lr, weight_decay=wd)
    else:
        optimizer = keras.optimizers.Adam(learning_rate=lr)

    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy")
    return model


NeuralSequenceRecommender._build_model = _patched_build_model
print("Patched GRU models to run with use_cudnn=False.")

In [ ]:
GENRE_COLUMNS = [
    "unknown",
    "Action",
    "Adventure",
    "Animation",
    "Children",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "FilmNoir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "SciFi",
    "Thriller",
    "War",
    "Western",
]


def download_movielens_100k(data_dir: Path, dataset_url: str) -> tuple[Path, Path]:
    data_dir.mkdir(parents=True, exist_ok=True)
    udata_path = data_dir / "ml-100k" / "u.data"
    uitem_path = data_dir / "ml-100k" / "u.item"

    if udata_path.exists() and uitem_path.exists():
        return udata_path, uitem_path

    archive_name = dataset_url.rsplit("/", 1)[-1]
    archive_path = data_dir / archive_name

    if not archive_path.exists():
        print("Downloading:", dataset_url)
        urllib.request.urlretrieve(dataset_url, archive_path)

    print("Extracting:", archive_path)
    if archive_path.suffix == ".zip":
        with zipfile.ZipFile(archive_path, "r") as zip_ref:
            zip_ref.extractall(data_dir)
    elif archive_name.endswith(".tar.gz") or archive_name.endswith(".tgz"):
        with tarfile.open(archive_path, "r:gz") as tar_ref:
            tar_ref.extractall(data_dir)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path}")

    if not udata_path.exists() or not uitem_path.exists():
        raise FileNotFoundError("MovieLens-100K files were not found after extraction.")

    return udata_path, uitem_path


def load_movielens_interactions(udata_path: Path) -> pd.DataFrame:
    return pd.read_csv(
        udata_path,
        sep="\t",
        names=["UserId", "ItemId", "Rating", "Timestamp"],
        engine="python",
    )


def sessionize(
    interactions: pd.DataFrame,
    session_gap_minutes: int,
    min_session_len: int,
) -> pd.DataFrame:
    if min_session_len < 2:
        raise ValueError("min_session_len must be >= 2 for next-item evaluation.")

    gap_seconds = session_gap_minutes * 60

    df = interactions.copy()
    df["Timestamp"] = df["Timestamp"].astype(int)
    df["ItemId"] = df["ItemId"].astype(int)
    df["Rating"] = df["Rating"].astype(float)

    df = df.sort_values(["UserId", "Timestamp", "ItemId"]).reset_index(drop=True)

    user_changed = df["UserId"].ne(df["UserId"].shift(1))
    ts_gap = df["Timestamp"].diff().fillna(0)
    new_session = user_changed | (ts_gap > gap_seconds)
    df["SessionId"] = new_session.cumsum().astype(int)

    session_sizes = df.groupby("SessionId").size()
    valid_sessions = session_sizes[session_sizes >= min_session_len].index
    df = df[df["SessionId"].isin(valid_sessions)].copy()

    new_ids = {old_id: idx + 1 for idx, old_id in enumerate(sorted(df["SessionId"].unique()))}
    df["SessionId"] = df["SessionId"].map(new_ids).astype(int)

    df["Time"] = pd.to_datetime(df["Timestamp"], unit="s", utc=True).dt.tz_localize(None)
    df["Time"] = df["Time"].dt.strftime("%Y-%m-%d %H:%M:%S.%f")
    df["Reward"] = df["Rating"].astype(float)

    out_df = df[["SessionId", "ItemId", "Time", "Reward"]]
    out_df = out_df.sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True)
    return out_df


def load_movielens_item_metadata(uitem_path: Path) -> pd.DataFrame:
    columns = [
        "ItemId",
        "Title",
        "ReleaseDate",
        "VideoReleaseDate",
        "IMDbURL",
        *GENRE_COLUMNS,
    ]
    return pd.read_csv(
        uitem_path,
        sep="|",
        names=columns,
        usecols=list(range(len(columns))),
        encoding="latin-1",
        engine="python",
    )


def build_genre_embedding_tables(
    item_metadata: pd.DataFrame,
    item_ids: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    filtered = item_metadata[item_metadata["ItemId"].isin(item_ids)].copy()
    if filtered.empty:
        raise ValueError("No overlapping item ids found for item metadata.")

    genre_matrix = filtered[GENRE_COLUMNS].to_numpy(dtype=np.float32)

    no_genre = (genre_matrix.sum(axis=1, keepdims=True) == 0).astype(np.float32)
    embeddings = np.concatenate([genre_matrix, no_genre], axis=1)

    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    embeddings = embeddings / norms

    item_data = filtered[["ItemId"]].copy()
    item_data["embedding"] = [row.astype(np.float32) for row in embeddings]
    item_data["class"] = np.argmax(embeddings, axis=1).astype(int)
    item_data["category_size"] = [[] for _ in range(len(item_data))]

    embeddings_csv_df = item_data[["ItemId", "embedding", "class"]].copy()
    embeddings_csv_df["embedding"] = embeddings_csv_df["embedding"].apply(
        lambda vec: json.dumps([float(x) for x in vec.tolist()])
    )

    return item_data, embeddings_csv_df


def build_dataset(
    sessions_csv_path: Path,
    dataset_pickle_path: Path,
) -> SessionDataset:
    dataset = SessionDataset(str(sessions_csv_path), n_withheld=1, evolving=False)
    split_strategy = TemporalSplit(
        test_frac=TEST_FRAC,
        num_folds=NUM_FOLDS,
        filter_non_trained_test_items=FILTER_NON_TRAINED_TEST_ITEMS,
        fold_strategy="chain",
    )
    dataset.load_and_split(split_strategy=split_strategy)
    dataset.to_pickle(str(dataset_pickle_path))
    return dataset

In [ ]:
udata_path, uitem_path = download_movielens_100k(DATA_DIR, DATASET_URL)
interactions = load_movielens_interactions(udata_path)
sessions_df = sessionize(
    interactions=interactions,
    session_gap_minutes=SESSION_GAP_MINUTES,
    min_session_len=MIN_SESSION_LEN,
)

sessions_csv_path = DATA_DIR / "sessions_ml100k.csv"
sessions_df.to_csv(sessions_csv_path, index=False)

dataset_pickle_path = DATA_DIR / "dataset_ml100k.pickle"
dataset = build_dataset(sessions_csv_path, dataset_pickle_path)

item_metadata = load_movielens_item_metadata(uitem_path)
item_ids = np.sort(dataset.get_input_data()["ItemId"].unique())
item_data, embeddings_csv_df = build_genre_embedding_tables(item_metadata, item_ids)
dataset.set_item_data(item_data)

embeddings_csv_path = DATA_DIR / "ml100k_item_embeddings.csv"
embeddings_csv_df.to_csv(embeddings_csv_path, index=False)

print("Sessions CSV:", sessions_csv_path)
print("Dataset pickle:", dataset_pickle_path)
print("Embeddings CSV:", embeddings_csv_path)
print("#Interactions:", len(dataset.get_input_data()))
print("#Sessions:", dataset.get_unique_sample_count())
print("#Unique items:", dataset.get_unique_item_count())

In [ ]:
def instantiate_models(embedding_csv_path: Path) -> dict[str, Any]:
    shared = {
        "cores": CORES,
        "is_verbose": VERBOSE,
    }
    neural_common = {
        "cores": CORES,
        "is_verbose": VERBOSE,
        "num_epochs": NUM_EPOCHS,
        "fit_batch_size": FIT_BATCH_SIZE,
        "pred_batch_size": PRED_BATCH_SIZE,
        "train_val_fraction": TRAIN_VAL_FRACTION,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "pred_seen": False,
    }
    optimizer_kwargs = {
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
    }

    models: dict[str, Any] = {
        "Popular": SessionBasedPopular(**shared),
        "SKNN": SessionBasedCF(
            k=SKNN_K,
            sample_size=SKNN_SAMPLE_SIZE,
            sampling="recent",
            similarity_measure="cosine",
            idf_weighting=True,
            filter_prompt_items=True,
            **shared,
        ),
        "S-SKNN": SessionBasedCF(
            k=SKNN_K,
            sample_size=SKNN_SAMPLE_SIZE,
            sampling="recent",
            similarity_measure="cosine",
            idf_weighting=True,
            sequential_weighting=True,
            filter_prompt_items=True,
            **shared,
        ),
        "SF-SKNN": SessionBasedCF(
            k=SKNN_K,
            sample_size=SKNN_SAMPLE_SIZE,
            sampling="recent",
            similarity_measure="cosine",
            idf_weighting=True,
            sequential_filter=True,
            filter_prompt_items=True,
            **shared,
        ),
        "V-SKNN": SessionBasedCF(
            k=SKNN_K,
            sample_size=SKNN_SAMPLE_SIZE,
            sampling="recent",
            similarity_measure="dot",
            idf_weighting=True,
            decay="harmonic",
            filter_prompt_items=True,
            **shared,
        ),
        "GRU4Rec": GRURec(
            N=MAX_SEQ_LEN,
            emb_dim=BASE_NEURAL_EMB_DIM,
            hidden_dim=HIDDEN_DIM,
            drop_rate=DROP_RATE,
            activation="relu",
            optimizer_kwargs=optimizer_kwargs,
            **neural_common,
        ),
        "BERT4Rec": BERT(
            N=MAX_SEQ_LEN,
            L=1,
            h=2,
            emb_dim=BASE_NEURAL_EMB_DIM,
            drop_rate=DROP_RATE,
            mask_prob=MASK_PROB,
            activation="gelu",
            optimizer_kwargs=optimizer_kwargs,
            transformer_layer_kwargs={"layout": "FDRN"},
            **neural_common,
        ),
        "SASRec": SASRec(
            N=MAX_SEQ_LEN,
            L=1,
            h=2,
            emb_dim=BASE_NEURAL_EMB_DIM,
            drop_rate=DROP_RATE,
            activation="relu",
            optimizer_kwargs=optimizer_kwargs,
            transformer_layer_kwargs={"layout": "NFDR"},
            **neural_common,
        ),
        "LLMSeqSim": LLMSeqSim(
            cores=CORES,
            is_verbose=VERBOSE,
            similarity_measure="cosine",
            embedding_combination_strategy="last",
            combination_decay=None,
            filter_prompt_items=True,
            batch_size=PRED_BATCH_SIZE,
            dim_reduction_config=None,
        ),
        "LLM2GRU4Rec": GRURecWithEmbeddings(
            product_embeddings_location=str(embedding_csv_path),
            red_method="PCA",
            red_params={},
            N=MAX_SEQ_LEN,
            emb_dim=EMBED_NEURAL_EMB_DIM,
            hidden_dim=HIDDEN_DIM,
            drop_rate=DROP_RATE,
            activation="relu",
            optimizer_kwargs=optimizer_kwargs,
            **neural_common,
        ),
        "LLM2BERT4Rec": BERTWithEmbeddings(
            product_embeddings_location=str(embedding_csv_path),
            red_method="PCA",
            red_params={},
            N=MAX_SEQ_LEN,
            L=1,
            h=2,
            emb_dim=EMBED_NEURAL_EMB_DIM,
            drop_rate=DROP_RATE,
            mask_prob=MASK_PROB,
            activation="gelu",
            optimizer_kwargs=optimizer_kwargs,
            transformer_layer_kwargs={"layout": "FDRN"},
            **neural_common,
        ),
        "LLM2SASRec": SASRecWithEmbeddings(
            product_embeddings_location=str(embedding_csv_path),
            red_method="PCA",
            red_params={},
            N=MAX_SEQ_LEN,
            L=1,
            h=2,
            emb_dim=EMBED_NEURAL_EMB_DIM,
            drop_rate=DROP_RATE,
            activation="relu",
            optimizer_kwargs=optimizer_kwargs,
            transformer_layer_kwargs={"layout": "NFDR"},
            **neural_common,
        ),
    }

    return models


models = instantiate_models(embeddings_csv_path)
print("Available models:", sorted(models.keys()))

In [ ]:
metric_list = [
    NormalizedDiscountedCumulativeGain(),
    HitRate(),
    MeanReciprocalRank(),
    CatalogCoverage(),
]
dependencies = {
    MetricDependency.NUM_ITEMS: dataset.get_unique_item_count(),
}


def train_and_predict(model: Any, dataset: SessionDataset, item_data: pd.DataFrame, top_k: int) -> dict[int, np.ndarray]:
    train_data = dataset.get_train_data()
    test_prompts = dataset.get_test_prompts()

    if isinstance(model, LLMSeqSim):
        model.train(train_data, item_data)
    elif isinstance(model, SessionBasedCF) and model.use_item_embeddings:
        model.train(train_data, item_data)
    else:
        model.train(train_data)

    return model.predict(test_prompts, top_k=top_k)


results_rows: list[dict[str, Any]] = []
failure_rows: list[dict[str, Any]] = []

for requested_name in INCLUDE_MODELS:
    if requested_name not in models:
        failure_rows.append({
            "RequestedModel": requested_name,
            "ErrorType": "UnknownModel",
            "Error": "Model is not present in instantiate_models().",
        })
        print(f"SKIP {requested_name}: not available")
        continue

    model = models[requested_name]
    print(f"\n=== Running {requested_name} ===")

    try:
        start = time.perf_counter()
        recommendations = train_and_predict(model, dataset, item_data, top_k=max(TOP_KS))
        elapsed = time.perf_counter() - start

        recs_path = RECS_DIR / f"recs_{requested_name}.pickle"
        with open(recs_path, "wb") as write_file:
            pickle.dump(recommendations, write_file)

        row: dict[str, Any] = {
            "RequestedModel": requested_name,
            "Model": model.name(),
            "TrainPredictSeconds": round(elapsed, 4),
        }

        for top_k in TOP_KS:
            report = Evaluation.eval(
                predictions=recommendations,
                ground_truths=dataset.get_test_ground_truths(),
                top_k=top_k,
                metrics=metric_list,
                metrics_per_sample=False,
                dependencies=dependencies,
                cores=EVAL_CORES,
                model_name=model.name(),
            )
            for metric_name, value in report.results.items():
                row[metric_name] = float(value)

        results_rows.append(row)
        print(f"OK {requested_name} -> {model.name()} ({elapsed:.2f}s)")
    except Exception as exc:
        failure_rows.append({
            "RequestedModel": requested_name,
            "ErrorType": type(exc).__name__,
            "Error": str(exc),
        })
        print(f"FAILED {requested_name}: {type(exc).__name__}: {exc}")

results_df = pd.DataFrame(results_rows)
if not results_df.empty:
    sort_col = "NDCG@20" if "NDCG@20" in results_df.columns else results_df.columns[-1]
    results_df = results_df.sort_values(sort_col, ascending=False).reset_index(drop=True)

failures_df = pd.DataFrame(failure_rows)

print("\n=== Results Preview ===")
display(results_df)
print("\n=== Failures Preview ===")
display(failures_df)

In [ ]:
results_csv = RESULTS_DIR / "core_model_results_ml100k.csv"
failures_csv = RESULTS_DIR / "core_model_failures_ml100k.csv"

results_df.to_csv(results_csv, index=False)
failures_df.to_csv(failures_csv, index=False)

print("Saved results to:", results_csv)
print("Saved failures to:", failures_csv)
print("Saved recommendation pickles to:", RECS_DIR)

## Notes

- LLM-style models in this notebook use genre-derived embeddings, not external OpenAI/PaLM embeddings.
- You can reduce runtime by lowering NUM_EPOCHS or shrinking INCLUDE_MODELS.
- Results are written under kaggle/artifacts/ml100k_core_models_single_notebook/results.